# De-Id SLM — submission publish (demo + HF push)

Run this on a **Colab GPU runtime** (Runtime → Change runtime type → GPU) to finish the
submission package:

1. **Demo** — prompted base vs. fine-tune side-by-side (screen-record this cell for the 3–5 min video).
2. **Push model** to the HuggingFace Hub (adapter + `MODEL_CARD.md`).
3. **Push dataset** to the Hub (train/val splits + card) — hard-ceiling guarded (0 eval leakage).

The BrainLift verdict + eval results are already committed. See `docs/submission-runbook.md`.

**Prereqs:** the `sft-v3-gpt551` adapter dir + v3 splits on Drive, and a HuggingFace write token.

## 0. Setup — clone, install, mount Drive, set paths

**Edit the three paths** to match your Drive before running.

In [ ]:
!git clone -b worktree-ship-submission https://github.com/f15cubing/slm-deid.git
%cd slm-deid
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

# --- EDIT THESE THREE ---
ADAPTER = "/content/drive/MyDrive/slm-deid-gpt551/sft-v3"      # canonical gpt551 LoRA adapter dir
SPLITS  = "/content/drive/MyDrive/slm-deid-gpt551/splits"     # dir with train.jsonl + val.jsonl
REPO_ID = "YOUR_HF_USERNAME/slm-deid-name-judgment"          # your HuggingFace repo id (created on push)
# ------------------------

import os
os.environ['ADAPTER'], os.environ['SPLITS'], os.environ['REPO_ID'] = ADAPTER, SPLITS, REPO_ID
assert os.path.isdir(ADAPTER), f'adapter dir not found: {ADAPTER}'
print('adapter files:', sorted(os.listdir(ADAPTER)))
print('splits files :', sorted(os.listdir(SPLITS)) if os.path.isdir(SPLITS) else 'MISSING')

## 1. Demo — base vs. tuned (record this for the video)

Loads the prompted base and the fine-tune, tags the ambiguous showcase passages with both, and
prints them side-by-side. Narrate the **“models disagree”** rows — that's the whole thesis on screen.

In [ ]:
!python -m src.demo --adapter "$ADAPTER"

## 2. Push the model to the Hub

Log in with a **write** token, dry-run (runs the eval-leak guard, uploads nothing), then push.

In [ ]:
# Paste your WRITE token (huggingface.co/settings/tokens) into the box.
# The token is stored so the push scripts below pick it up automatically.
from huggingface_hub import login
login()

In [ ]:
!python scripts/push_to_hub.py --adapter "$ADAPTER" --repo-id "$REPO_ID" --dry-run

In [ ]:
!python scripts/push_to_hub.py --adapter "$ADAPTER" --repo-id "$REPO_ID"

## 3. Push the dataset to the Hub (hard-ceiling guarded)

The dry-run asserts **0 `quarantine=true` rows + 0 overlap** vs the quarantined eval inputs and
uploads nothing. Only run the real push if it prints `Leak-guard PASSED`.

> If your Drive splits are the authored v3 (924/102) rather than gpt551 (818/90), that's fine — the
> guard passes either way; just note which you published in the dataset card.

In [ ]:
!python scripts/push_dataset.py --splits-dir "$SPLITS" --repo-id "$REPO_ID" --dry-run

In [ ]:
!python scripts/push_dataset.py --splits-dir "$SPLITS" --repo-id "$REPO_ID"

## 4. Collect the submission links

- Model:   `https://huggingface.co/<REPO_ID>`
- Dataset: `https://huggingface.co/datasets/<REPO_ID>`
- Demo video (record cell 1)
- Repo:    `https://github.com/f15cubing/slm-deid`

Drop these into the README header. The BrainLift verdict (checklist #4) and eval results
(checklist #3) are already committed.